# Chapter 9: Machine Translation

Machine translation (MT) is a subfield of natural language processing (NLP) that focuses on the automatic translation of text or speech from one language to another. With the rise of globalization and the internet, the demand for efficient and accurate translation systems has grown significantly.

This chapter explores a variety of techniques and models used in machine translation:

1. **Sequence to Sequence (Seq2Seq) Models** -- The foundational encoder-decoder architecture
2. **Attention Mechanisms** -- Enhancing Seq2Seq models by allowing the decoder to focus on relevant input parts
3. **Transformer Models** -- The modern architecture that processes sequences in parallel using self-attention

By the end of this chapter, you will understand how modern machine translation systems work, the challenges they address, and how to implement them using popular NLP libraries.

## Setup

Install and import the required packages.

In [ ]:
!pip install tensorflow transformers torch matplotlib seaborn --quiet

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate, TimeDistributed
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

---
## 9.1 Sequence to Sequence Models

### 9.1.1 Understanding Sequence to Sequence Models

Sequence to sequence (Seq2Seq) models are a type of neural network architecture designed for tasks where the input and output are sequences of different lengths. Originally developed for machine translation, Seq2Seq models have since been applied to text summarization, speech recognition, and chatbot development.

A Seq2Seq model consists of two main components:

**Encoder:**
- Processes the input sequence and converts it into a fixed-size **context vector** (also called the hidden state or thought vector)
- Typically uses RNN cells such as LSTM or GRU units
- Each element of the input sequence is processed one at a time, updating the hidden state
- The final hidden state contains a compressed representation of the entire input sequence

**Decoder:**
- Generates the output sequence from the context vector provided by the encoder
- Works token by token: receives the context vector and a start token, produces the first output token
- Each generated token is fed back as input to produce the next token
- Continues until an end token is generated or a maximum length is reached

```
Input Sequence --> [Encoder] --> Context Vector --> [Decoder] --> Output Sequence
```

### 9.1.2 Implementing a Basic Seq2Seq Model

We will use TensorFlow to implement a basic Seq2Seq model for translating simple English phrases to French.

In [ ]:
# Sample data: English to French
input_texts = [
    "Hello.",
    "How are you?",
    "What is your name?",
    "Good morning.",
    "Good night."
]

target_texts = [
    "Bonjour.",
    "Comment \u00e7a va?",
    "Quel est votre nom?",
    "Bonjour.",
    "Bonne nuit."
]

print("Sample parallel corpus:")
for en, fr in zip(input_texts, target_texts):
    print(f"  {en:25s} -> {fr}")

In [ ]:
# Tokenize the data
input_tokenizer = Tokenizer()
input_tokenizer.fit_on_texts(input_texts)
input_sequences = input_tokenizer.texts_to_sequences(input_texts)
input_maxlen = max(len(seq) for seq in input_sequences)
input_vocab_size = len(input_tokenizer.word_index) + 1

target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(target_texts)
target_sequences = target_tokenizer.texts_to_sequences(target_texts)
target_maxlen = max(len(seq) for seq in target_sequences)
target_vocab_size = len(target_tokenizer.word_index) + 1

print(f"Input vocab size: {input_vocab_size}, max length: {input_maxlen}")
print(f"Target vocab size: {target_vocab_size}, max length: {target_maxlen}")
print(f"\nInput word index: {input_tokenizer.word_index}")
print(f"Target word index: {target_tokenizer.word_index}")

In [ ]:
# Pad sequences to ensure uniform length
input_sequences = pad_sequences(input_sequences, maxlen=input_maxlen, padding='post')
target_sequences = pad_sequences(target_sequences, maxlen=target_maxlen, padding='post')

# Split target sequences into input and output for teacher forcing
# Decoder input: all tokens except the last
# Decoder output: all tokens except the first (shifted by one position)
target_input_sequences = target_sequences[:, :-1]
target_output_sequences = target_sequences[:, 1:]

print("Padded input sequences:\n", input_sequences)
print("\nTarget input sequences (decoder input):\n", target_input_sequences)
print("\nTarget output sequences (decoder output):\n", target_output_sequences)

In [ ]:
# Build the Seq2Seq model
latent_dim = 256

# --- Encoder ---
encoder_inputs = Input(shape=(input_maxlen,))
encoder_embedding = tf.keras.layers.Embedding(input_vocab_size, latent_dim)(encoder_inputs)
encoder_lstm = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# --- Decoder ---
decoder_inputs = Input(shape=(None,))
decoder_embedding = tf.keras.layers.Embedding(target_vocab_size, latent_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(target_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Define the full training model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model.summary()

In [ ]:
# Train the model
model.fit(
    [input_sequences, target_input_sequences],
    target_output_sequences,
    batch_size=64,
    epochs=100,
    validation_split=0.2
)

#### Inference: Translating New Sentences

For inference, we need separate encoder and decoder models. The encoder produces the context vector, and the decoder generates tokens one at a time.

In [ ]:
# --- Inference Models ---

# Encoder model: takes input and returns hidden states
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder model: takes previous states and token, returns prediction and new states
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)

In [ ]:
def decode_sequence(input_seq):
    """Translate an input sequence using the trained Seq2Seq model."""
    # Encode the input as state vectors
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1
    target_seq = np.zeros((1, 1))
    # Use 'bonjour' as the start token
    target_seq[0, 0] = target_tokenizer.word_index['bonjour']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value)

        # Sample the token with highest probability
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = target_tokenizer.index_word[sampled_token_index]
        decoded_sentence += ' ' + sampled_word

        # Exit condition: hit max length or find stop token
        if (sampled_word == '.' or len(decoded_sentence) > target_maxlen):
            stop_condition = True

        # Update the target sequence (length 1)
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence


# Test the model on training data
print("=" * 50)
print("Seq2Seq Translation Results")
print("=" * 50)
for seq_index in range(5):
    input_seq = input_sequences[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print(f"Input:   {input_texts[seq_index]}")
    print(f"Output:  {decoded_sentence.strip()}")
    print("-" * 50)

### 9.1.3 Advantages and Limitations of Seq2Seq Models

**Advantages:**
- **Flexible Architecture:** Can handle sequences of varying lengths for both input and output, making them suitable for translation, summarization, speech recognition, and more.
- **Captures Context:** The encoder generates a context vector that encapsulates essential information from the input sequence.
- **End-to-End Training:** The entire model is optimized simultaneously, simplifying the training process.

**Limitations:**
- **Fixed-Length Context Vector:** The single context vector can become a bottleneck for long input sequences, leading to information loss.
- **Training Complexity:** Requires large datasets and significant computational resources (GPUs/TPUs).
- **Exposure Bias:** During training the model sees ground truth, but during inference it uses its own predictions, leading to error accumulation.
- **Limited Interpretability:** Like many deep learning models, Seq2Seq models are "black boxes" that are difficult to interpret.

---
## 9.2 Attention Mechanisms

### 9.2.1 Understanding Attention Mechanisms

Attention mechanisms address one of the major limitations of traditional Seq2Seq models: the **fixed-length context vector**. Instead of relying on a single, static context vector, the decoder dynamically generates context vectors that emphasize the most relevant parts of the input sequence at each step.

This significantly improves the model's ability to handle long and complex input sequences.

### 9.2.2 How Attention Mechanisms Work

The attention mechanism operates in several steps:

1. **Compute Attention Scores:** Calculate a score for each encoder hidden state measuring its relevance to the current decoder state. Common methods include:
   - *Dot-Product Attention:* Dot product of encoder and decoder hidden states (simple, efficient)
   - *Additive (Bahdanau) Attention:* Concatenate states and pass through a feed-forward network (more flexible)

2. **Calculate Attention Weights:** Apply a softmax function to the scores, producing a probability distribution that sums to 1.

3. **Generate Context Vector:** Compute a weighted sum of the encoder hidden states using the attention weights. This creates a context vector tailored to each decoding step.

4. **Update Decoder State:** Combine the context vector with the current decoder hidden state to generate the next output token.

```
Encoder Hidden States: [h1, h2, h3, ..., hn]
                         |   |   |        |
Attention Weights:     [a1, a2, a3, ..., an]  (sum = 1)
                         \   \   \       /
Context Vector:          weighted sum
                              |
                         [Decoder] --> next token
```

### 9.2.3 Implementing Attention Mechanisms in Seq2Seq Models

We will enhance the Seq2Seq model with an attention mechanism using TensorFlow's built-in `Attention` layer.

In [ ]:
# Re-prepare data (fresh tokenizers for this section)
input_texts_attn = [
    "Hello.",
    "How are you?",
    "What is your name?",
    "Good morning.",
    "Good night."
]

target_texts_attn = [
    "Bonjour.",
    "Comment \u00e7a va?",
    "Quel est votre nom?",
    "Bonjour.",
    "Bonne nuit."
]

# Tokenize
input_tokenizer_attn = Tokenizer()
input_tokenizer_attn.fit_on_texts(input_texts_attn)
input_sequences_attn = input_tokenizer_attn.texts_to_sequences(input_texts_attn)
input_maxlen_attn = max(len(seq) for seq in input_sequences_attn)
input_vocab_size_attn = len(input_tokenizer_attn.word_index) + 1

target_tokenizer_attn = Tokenizer()
target_tokenizer_attn.fit_on_texts(target_texts_attn)
target_sequences_attn = target_tokenizer_attn.texts_to_sequences(target_texts_attn)
target_maxlen_attn = max(len(seq) for seq in target_sequences_attn)
target_vocab_size_attn = len(target_tokenizer_attn.word_index) + 1

# Pad sequences
input_sequences_attn = pad_sequences(input_sequences_attn, maxlen=input_maxlen_attn, padding='post')
target_sequences_attn = pad_sequences(target_sequences_attn, maxlen=target_maxlen_attn, padding='post')

# Split target sequences
target_input_sequences_attn = target_sequences_attn[:, :-1]
target_output_sequences_attn = target_sequences_attn[:, 1:]

print(f"Input vocab size: {input_vocab_size_attn}, max length: {input_maxlen_attn}")
print(f"Target vocab size: {target_vocab_size_attn}, max length: {target_maxlen_attn}")

In [ ]:
# Define the Seq2Seq model WITH Attention
latent_dim_attn = 256

# --- Encoder ---
encoder_inputs_attn = Input(shape=(input_maxlen_attn,))
encoder_embedding_attn = Embedding(input_vocab_size_attn, latent_dim_attn)(encoder_inputs_attn)
encoder_lstm_attn = LSTM(latent_dim_attn, return_sequences=True, return_state=True)
# NOTE: return_sequences=True so we get ALL hidden states (needed for attention)
encoder_outputs_attn, state_h_attn, state_c_attn = encoder_lstm_attn(encoder_embedding_attn)
encoder_states_attn = [state_h_attn, state_c_attn]

# --- Decoder ---
decoder_inputs_attn = Input(shape=(None,))
decoder_embedding_attn = Embedding(target_vocab_size_attn, latent_dim_attn)(decoder_inputs_attn)
decoder_lstm_attn = LSTM(latent_dim_attn, return_sequences=True, return_state=True)
decoder_outputs_attn, _, _ = decoder_lstm_attn(decoder_embedding_attn, initial_state=encoder_states_attn)

# --- Attention Mechanism ---
# Computes attention weights between decoder outputs and encoder outputs
attention_layer = tf.keras.layers.Attention()
attention_output = attention_layer([decoder_outputs_attn, encoder_outputs_attn])

# Concatenate attention output with decoder outputs
decoder_concat_input = Concatenate(axis=-1)([decoder_outputs_attn, attention_output])

# Dense layer to generate predictions
decoder_dense_attn = TimeDistributed(Dense(target_vocab_size_attn, activation='softmax'))
decoder_outputs_attn = decoder_dense_attn(decoder_concat_input)

# Define the model
model_attn = Model([encoder_inputs_attn, decoder_inputs_attn], decoder_outputs_attn)

# Compile
model_attn.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

model_attn.summary()

In [ ]:
# Train the attention model
model_attn.fit(
    [input_sequences_attn, target_input_sequences_attn],
    target_output_sequences_attn,
    batch_size=64,
    epochs=100,
    validation_split=0.2
)

In [ ]:
# --- Inference Models for Attention Seq2Seq ---

# Encoder model: returns encoder outputs AND states
encoder_model_attn = Model(encoder_inputs_attn, [encoder_outputs_attn] + encoder_states_attn)

# Decoder model
decoder_state_input_h_attn = Input(shape=(latent_dim_attn,))
decoder_state_input_c_attn = Input(shape=(latent_dim_attn,))
decoder_states_inputs_attn = [decoder_state_input_h_attn, decoder_state_input_c_attn]
decoder_hidden_state_input = Input(shape=(input_maxlen_attn, latent_dim_attn))

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm_attn(
    decoder_embedding_attn, initial_state=decoder_states_inputs_attn)

attention_output_inf = attention_layer([decoder_outputs_inf, decoder_hidden_state_input])
decoder_concat_inf = Concatenate(axis=-1)([decoder_outputs_inf, attention_output_inf])
decoder_outputs_inf = decoder_dense_attn(decoder_concat_inf)

decoder_model_attn = Model(
    [decoder_inputs_attn] + [decoder_hidden_state_input] + decoder_states_inputs_attn,
    [decoder_outputs_inf] + [state_h_inf, state_c_inf])

In [ ]:
def decode_sequence_attn(input_seq):
    """Translate an input sequence using the Seq2Seq model with attention."""
    # Encode the input
    encoder_outs, state_h, state_c = encoder_model_attn.predict(input_seq)
    states_value = [state_h, state_c]

    # Start with 'bonjour' token
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = target_tokenizer_attn.word_index['bonjour']

    stop_condition = False
    decoded_sentence = ''

    while not stop_condition:
        output_tokens, h, c = decoder_model_attn.predict(
            [target_seq] + [encoder_outs] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = target_tokenizer_attn.index_word[sampled_token_index]
        decoded_sentence += ' ' + sampled_word

        if (sampled_word == '.' or len(decoded_sentence) > target_maxlen_attn):
            stop_condition = True

        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return decoded_sentence


# Test the attention model
print("=" * 50)
print("Seq2Seq with Attention Translation Results")
print("=" * 50)
for seq_index in range(5):
    input_seq = input_sequences_attn[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence_attn(input_seq)
    print(f"Input:   {input_texts_attn[seq_index]}")
    print(f"Output:  {decoded_sentence.strip()}")
    print("-" * 50)

### 9.2.4 Advantages and Limitations of Attention Mechanisms

**Advantages:**
- **Improved Performance:** The decoder focuses on the most relevant parts of the input at each step, producing more accurate translations.
- **Handling Long Sequences:** Directly accesses the entire input sequence at each decoding step, reducing information loss.
- **Flexibility:** Easily integrated with various architectures (RNNs, LSTMs, GRUs).

**Limitations:**
- **Complexity:** More computational resources are required (increased memory and processing power).
- **Training Time:** Computing attention scores and context vectors at each step adds to overall training time.

---
## 9.3 Transformer Models

### 9.3.1 Understanding Transformer Models

Transformer models, introduced by Vaswani et al. in *"Attention is All You Need"* (2017), revolutionized NLP by addressing limitations of RNNs and traditional attention mechanisms. Key innovations:

- **Self-attention mechanisms** process input sequences in parallel (not sequentially like RNNs)
- **Context-dependent weights** allow the model to focus on the most relevant tokens for each task
- **Multi-layer architecture** builds increasingly abstract representations

### 9.3.2 Architecture of Transformer Models

The transformer consists of an **Encoder** and **Decoder**, each composed of multiple identical layers containing:

1. **Multi-Head Self-Attention:** Multiple attention heads attend to different parts of the input simultaneously. Each head captures different aspects (e.g., one might focus on the subject, another on the verb). Outputs are concatenated and linearly transformed.

2. **Feed-Forward Neural Network (FFNN):** Two linear transformations with a ReLU activation between them. Applied independently to each position, refining the representations from the attention layer.

3. **Layer Normalization:** Normalizes inputs to each sub-layer, stabilizing training by maintaining consistent input scales.

4. **Residual Connections:** The original input of each sub-layer is added to its output, helping gradient flow and preventing vanishing/exploding gradients.

5. **Positional Encoding:** Since transformers process tokens in parallel, positional encoding adds position information to each token so the model understands sequential order.

### 9.3.3 Implementing Transformer Models with Hugging Face

We will use the Hugging Face `transformers` library with the T5 (Text-To-Text Transfer Transformer) model, which has been pre-trained on a variety of text generation tasks including translation.

In [ ]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Load the pre-trained T5 model and tokenizer
model_name = "t5-small"
t5_model = T5ForConditionalGeneration.from_pretrained(model_name)
t5_tokenizer = T5Tokenizer.from_pretrained(model_name)

print(f"Loaded model: {model_name}")
print(f"Model parameters: {sum(p.numel() for p in t5_model.parameters()):,}")

In [ ]:
# T5 uses a text-to-text format: prefix the input with the task description
text = """translate English to French: Machine learning is a subset of artificial \
intelligence. It involves algorithms and statistical models to perform tasks without \
explicit instructions. Machine learning is widely used in various applications such \
as image recognition, natural language processing, and autonomous driving. It relies \
on patterns and inference instead of predefined rules."""

# Tokenize and encode the text
inputs = t5_tokenizer.encode(text, return_tensors="pt", max_length=512, truncation=True)
print(f"Input token count: {inputs.shape[1]}")

# Generate the translation using beam search
output_ids = t5_model.generate(inputs, max_length=150, num_beams=4, early_stopping=True)
translation = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\nTranslation:")
print(translation)

### 9.3.4 Visualizing Self-Attention Scores

We can visualize the self-attention scores to understand how the model focuses on different parts of the input sequence when generating each output token. This provides insight into the model's internal decision-making.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_attention(model, tokenizer, text):
    """Visualize self-attention scores from a transformer model."""
    inputs = tokenizer.encode(text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs, output_attentions=True)
    attentions = outputs[-1]  # Get the attention scores

    # Convert to numpy array for visualization
    attention_matrix = attentions[-1][0][0].detach().numpy()

    # Plot the attention scores as a heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(attention_matrix, cmap="viridis")
    plt.title("Self-Attention Scores")
    plt.xlabel("Input Tokens")
    plt.ylabel("Output Tokens")
    plt.show()

# Visualize attention scores for a sample sentence
sample_text = "translate English to French: How are you?"
visualize_attention(t5_model, t5_tokenizer, sample_text)

The heatmap above shows how the model attends to different input tokens when generating each output token. Brighter colors indicate higher attention weights -- these are the tokens the model considers most important for producing a given output token.

### 9.3.5 Advantages and Limitations of Transformer Models

**Advantages:**
- **Parallel Processing:** Transformers process input sequences in parallel (unlike RNNs which are sequential), significantly reducing training time.
- **Long-Range Dependencies:** Self-attention can attend to all positions simultaneously, effectively capturing long-range dependencies.
- **State-of-the-Art Performance:** Models like BERT, GPT, and T5 have set new benchmarks across virtually all NLP tasks.

**Limitations:**
- **Computational Resources:** Training large transformer models requires powerful hardware (GPUs/TPUs) and significant memory.
- **Complexity:** The architecture (multi-head attention, positional encoding, etc.) is more complex than traditional RNNs, requiring deeper understanding to implement and tune effectively.

---
## Model Comparison

| Feature | Seq2Seq | Seq2Seq + Attention | Transformer |
|---|---|---|---|
| **Context handling** | Fixed-size context vector | Dynamic context at each step | Full self-attention over all tokens |
| **Long sequences** | Information loss for long inputs | Much better via attention weights | Excellent via parallel self-attention |
| **Processing** | Sequential (RNN-based) | Sequential (RNN-based) | Parallel |
| **Training speed** | Moderate | Slower (attention overhead) | Fast (parallelization) |
| **Performance** | Baseline | Improved | State-of-the-art |
| **Complexity** | Simple | Moderate | High |
| **Resources needed** | Moderate | Moderate-High | High |

---
## Practical Exercises

### Exercise 1: Seq2Seq Model -- English to Spanish

**Task:** Implement a basic Seq2Seq model to translate simple English phrases to Spanish.

Use the following sample data:
- Input: `["Hello.", "How are you?", "What is your name?", "Good morning.", "Good night."]`
- Target: `["Hola.", "\u00bfC\u00f3mo est\u00e1s?", "\u00bfCu\u00e1l es tu nombre?", "Buenos d\u00edas.", "Buenas noches."]`

In [ ]:
# Exercise 1 Solution: Seq2Seq English to Spanish

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample data
input_texts_ex1 = ["Hello.", "How are you?", "What is your name?", "Good morning.", "Good night."]
target_texts_ex1 = ["Hola.", "\u00bfC\u00f3mo est\u00e1s?", "\u00bfCu\u00e1l es tu nombre?", "Buenos d\u00edas.", "Buenas noches."]

# Tokenize
input_tok_ex1 = Tokenizer()
input_tok_ex1.fit_on_texts(input_texts_ex1)
input_seq_ex1 = input_tok_ex1.texts_to_sequences(input_texts_ex1)
input_maxlen_ex1 = max(len(s) for s in input_seq_ex1)
input_vocab_ex1 = len(input_tok_ex1.word_index) + 1

target_tok_ex1 = Tokenizer()
target_tok_ex1.fit_on_texts(target_texts_ex1)
target_seq_ex1 = target_tok_ex1.texts_to_sequences(target_texts_ex1)
target_maxlen_ex1 = max(len(s) for s in target_seq_ex1)
target_vocab_ex1 = len(target_tok_ex1.word_index) + 1

# Pad
input_seq_ex1 = pad_sequences(input_seq_ex1, maxlen=input_maxlen_ex1, padding='post')
target_seq_ex1 = pad_sequences(target_seq_ex1, maxlen=target_maxlen_ex1, padding='post')

target_input_ex1 = target_seq_ex1[:, :-1]
target_output_ex1 = target_seq_ex1[:, 1:]

# Build model
latent_dim_ex1 = 256

enc_inputs_ex1 = Input(shape=(input_maxlen_ex1,))
enc_emb_ex1 = Embedding(input_vocab_ex1, latent_dim_ex1)(enc_inputs_ex1)
enc_lstm_ex1 = LSTM(latent_dim_ex1, return_state=True)
enc_out_ex1, sh_ex1, sc_ex1 = enc_lstm_ex1(enc_emb_ex1)
enc_states_ex1 = [sh_ex1, sc_ex1]

dec_inputs_ex1 = Input(shape=(None,))
dec_emb_ex1 = Embedding(target_vocab_ex1, latent_dim_ex1)(dec_inputs_ex1)
dec_lstm_ex1 = LSTM(latent_dim_ex1, return_sequences=True, return_state=True)
dec_out_ex1, _, _ = dec_lstm_ex1(dec_emb_ex1, initial_state=enc_states_ex1)
dec_dense_ex1 = Dense(target_vocab_ex1, activation='softmax')
dec_out_ex1 = dec_dense_ex1(dec_out_ex1)

model_ex1 = Model([enc_inputs_ex1, dec_inputs_ex1], dec_out_ex1)
model_ex1.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Train
model_ex1.fit([input_seq_ex1, target_input_ex1], target_output_ex1,
              batch_size=64, epochs=100, validation_split=0.2)

# Inference models
enc_model_ex1 = Model(enc_inputs_ex1, enc_states_ex1)

dec_sh_in = Input(shape=(latent_dim_ex1,))
dec_sc_in = Input(shape=(latent_dim_ex1,))
dec_states_in_ex1 = [dec_sh_in, dec_sc_in]
dec_out_inf_ex1, sh_inf, sc_inf = dec_lstm_ex1(dec_emb_ex1, initial_state=dec_states_in_ex1)
dec_out_inf_ex1 = dec_dense_ex1(dec_out_inf_ex1)
dec_model_ex1 = Model([dec_inputs_ex1] + dec_states_in_ex1, [dec_out_inf_ex1, sh_inf, sc_inf])

def decode_ex1(input_seq):
    states = enc_model_ex1.predict(input_seq)
    tgt = np.zeros((1, 1))
    tgt[0, 0] = target_tok_ex1.word_index['hola']
    decoded = ''
    for _ in range(target_maxlen_ex1 + 5):
        out, h, c = dec_model_ex1.predict([tgt] + states)
        idx = np.argmax(out[0, -1, :])
        word = target_tok_ex1.index_word[idx]
        decoded += ' ' + word
        if word == '.' or len(decoded) > target_maxlen_ex1:
            break
        tgt = np.zeros((1, 1))
        tgt[0, 0] = idx
        states = [h, c]
    return decoded

# Test
print("=" * 50)
print("Exercise 1: Seq2Seq English -> Spanish")
print("=" * 50)
for i in range(5):
    seq = input_seq_ex1[i:i+1]
    print(f"Input:  {input_texts_ex1[i]}")
    print(f"Output: {decode_ex1(seq).strip()}")
    print("-" * 50)

### Exercise 2: Seq2Seq with Attention -- English to Spanish

**Task:** Enhance the Seq2Seq model from Exercise 1 with an attention mechanism.

In [ ]:
# Exercise 2 Solution: Seq2Seq with Attention, English to Spanish

from tensorflow.keras.layers import Concatenate, TimeDistributed

# Re-use data from Exercise 1
# Re-tokenize for clean state
input_tok_ex2 = Tokenizer()
input_tok_ex2.fit_on_texts(input_texts_ex1)
input_seq_ex2 = pad_sequences(input_tok_ex2.texts_to_sequences(input_texts_ex1),
                               maxlen=input_maxlen_ex1, padding='post')
input_vocab_ex2 = len(input_tok_ex2.word_index) + 1

target_tok_ex2 = Tokenizer()
target_tok_ex2.fit_on_texts(target_texts_ex1)
target_seq_ex2 = pad_sequences(target_tok_ex2.texts_to_sequences(target_texts_ex1),
                                maxlen=target_maxlen_ex1, padding='post')
target_vocab_ex2 = len(target_tok_ex2.word_index) + 1

target_input_ex2 = target_seq_ex2[:, :-1]
target_output_ex2 = target_seq_ex2[:, 1:]

latent_dim_ex2 = 256

# Encoder
enc_in_ex2 = Input(shape=(input_maxlen_ex1,))
enc_emb_ex2 = Embedding(input_vocab_ex2, latent_dim_ex2)(enc_in_ex2)
enc_lstm_ex2 = LSTM(latent_dim_ex2, return_sequences=True, return_state=True)
enc_out_ex2, sh2, sc2 = enc_lstm_ex2(enc_emb_ex2)
enc_states_ex2 = [sh2, sc2]

# Decoder
dec_in_ex2 = Input(shape=(None,))
dec_emb_ex2 = Embedding(target_vocab_ex2, latent_dim_ex2)(dec_in_ex2)
dec_lstm_ex2 = LSTM(latent_dim_ex2, return_sequences=True, return_state=True)
dec_out_ex2, _, _ = dec_lstm_ex2(dec_emb_ex2, initial_state=enc_states_ex2)

# Attention
attn_ex2 = tf.keras.layers.Attention()([dec_out_ex2, enc_out_ex2])
dec_concat_ex2 = Concatenate(axis=-1)([dec_out_ex2, attn_ex2])
dec_dense_ex2 = TimeDistributed(Dense(target_vocab_ex2, activation='softmax'))
dec_out_ex2 = dec_dense_ex2(dec_concat_ex2)

model_ex2 = Model([enc_in_ex2, dec_in_ex2], dec_out_ex2)
model_ex2.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

# Train
model_ex2.fit([input_seq_ex2, target_input_ex2], target_output_ex2,
              batch_size=64, epochs=100, validation_split=0.2)

# Inference models
enc_model_ex2 = Model(enc_in_ex2, [enc_out_ex2] + enc_states_ex2)

dec_sh2_in = Input(shape=(latent_dim_ex2,))
dec_sc2_in = Input(shape=(latent_dim_ex2,))
dec_hidden_in = Input(shape=(input_maxlen_ex1, latent_dim_ex2))
dec_out2_inf, sh2_inf, sc2_inf = dec_lstm_ex2(dec_emb_ex2, initial_state=[dec_sh2_in, dec_sc2_in])
attn2_inf = tf.keras.layers.Attention()([dec_out2_inf, dec_hidden_in])
dec_concat2_inf = Concatenate(axis=-1)([dec_out2_inf, attn2_inf])
dec_out2_inf = dec_dense_ex2(dec_concat2_inf)

dec_model_ex2 = Model(
    [dec_in_ex2, dec_hidden_in, dec_sh2_in, dec_sc2_in],
    [dec_out2_inf, sh2_inf, sc2_inf])

def decode_ex2(input_seq):
    enc_outs, sh, sc = enc_model_ex2.predict(input_seq)
    states = [sh, sc]
    tgt = np.zeros((1, 1))
    tgt[0, 0] = target_tok_ex2.word_index['hola']
    decoded = ''
    for _ in range(target_maxlen_ex1 + 5):
        out, h, c = dec_model_ex2.predict([tgt, enc_outs] + states)
        idx = np.argmax(out[0, -1, :])
        word = target_tok_ex2.index_word[idx]
        decoded += ' ' + word
        if word == '.' or len(decoded) > target_maxlen_ex1:
            break
        tgt = np.zeros((1, 1))
        tgt[0, 0] = idx
        states = [h, c]
    return decoded

# Test
print("=" * 50)
print("Exercise 2: Seq2Seq + Attention English -> Spanish")
print("=" * 50)
for i in range(5):
    seq = input_seq_ex2[i:i+1]
    print(f"Input:  {input_texts_ex1[i]}")
    print(f"Output: {decode_ex2(seq).strip()}")
    print("-" * 50)

### Exercise 3: Transformer Model with T5 -- English to Spanish

**Task:** Use the T5 transformer model to translate English text to Spanish.

In [ ]:
# Exercise 3 Solution: T5 Transformer English to Spanish

from transformers import T5ForConditionalGeneration, T5Tokenizer

model_name = "t5-small"
t5_model_ex3 = T5ForConditionalGeneration.from_pretrained(model_name)
t5_tokenizer_ex3 = T5Tokenizer.from_pretrained(model_name)

text_ex3 = """translate English to Spanish: Machine learning is a subset of artificial \
intelligence. It involves algorithms and statistical models to perform tasks without \
explicit instructions. Machine learning is widely used in various applications such \
as image recognition, natural language processing, and autonomous driving. \
It relies on patterns and inference instead of predefined rules."""

inputs_ex3 = t5_tokenizer_ex3.encode(text_ex3, return_tensors="pt", max_length=512, truncation=True)
output_ids_ex3 = t5_model_ex3.generate(inputs_ex3, max_length=150, num_beams=4, early_stopping=True)
translation_ex3 = t5_tokenizer_ex3.decode(output_ids_ex3[0], skip_special_tokens=True)

print("Translation (English -> Spanish):")
print(translation_ex3)

### Exercise 4: Visualizing Attention Scores

**Task:** Visualize the self-attention scores of the T5 model for the input sentence: `"translate English to Spanish: How are you?"`

In [ ]:
# Exercise 4 Solution: Visualize T5 Attention for Spanish Translation

import matplotlib.pyplot as plt
import seaborn as sns

def visualize_attention_ex4(model, tokenizer, text):
    """Visualize self-attention scores from a transformer model."""
    inputs = tokenizer.encode(text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs, output_attentions=True)
    attentions = outputs[-1]

    attention_matrix = attentions[-1][0][0].detach().numpy()

    plt.figure(figsize=(10, 8))
    sns.heatmap(attention_matrix, cmap="viridis")
    plt.title("Self-Attention Scores (English to Spanish)")
    plt.xlabel("Input Tokens")
    plt.ylabel("Output Tokens")
    plt.show()

sample_text_ex4 = "translate English to Spanish: How are you?"
visualize_attention_ex4(t5_model_ex3, t5_tokenizer_ex3, sample_text_ex4)

### Exercise 5: Comparing Seq2Seq, Attention, and Transformer Models

**Task:** Compare the translations generated by all three models for the sentence: `"Good evening."`

In [ ]:
# Exercise 5 Solution: Model Comparison

test_sentence = "Good evening."

# Seq2Seq (Exercise 1)
test_seq = pad_sequences(
    input_tok_ex1.texts_to_sequences([test_sentence]),
    maxlen=input_maxlen_ex1, padding='post'
)
seq2seq_result = decode_ex1(test_seq)
print(f"Seq2Seq Translation:              {seq2seq_result.strip()}")

# Seq2Seq with Attention (Exercise 2)
test_seq_attn = pad_sequences(
    input_tok_ex2.texts_to_sequences([test_sentence]),
    maxlen=input_maxlen_ex1, padding='post'
)
attention_result = decode_ex2(test_seq_attn)
print(f"Seq2Seq + Attention Translation:   {attention_result.strip()}")

# Transformer (T5)
text_t5 = f"translate English to Spanish: {test_sentence}"
inputs_t5 = t5_tokenizer_ex3.encode(text_t5, return_tensors="pt", max_length=512, truncation=True)
output_t5 = t5_model_ex3.generate(inputs_t5, max_length=50, num_beams=4, early_stopping=True)
transformer_result = t5_tokenizer_ex3.decode(output_t5[0], skip_special_tokens=True)
print(f"Transformer (T5) Translation:      {transformer_result}")

print("\n" + "=" * 60)
print("Note: For simple sentences, all models may produce similar")
print("results. Differences become more apparent with longer,")
print("more complex sentences where attention and transformers")
print("can better capture long-range dependencies.")
print("=" * 60)

---
## Chapter Summary

In this chapter, we explored three major approaches to machine translation:

1. **Seq2Seq Models:** The foundational encoder-decoder architecture that compresses input into a fixed-size context vector. Simple and flexible, but limited by information loss on long sequences.

2. **Attention Mechanisms:** Enhanced Seq2Seq models that allow the decoder to dynamically focus on relevant parts of the input at each decoding step. Better handling of long sequences and improved translation accuracy.

3. **Transformer Models:** The modern architecture that uses self-attention to process sequences in parallel. State-of-the-art performance via models like T5, BERT, and GPT, with superior handling of long-range dependencies.

**Key Takeaway:** The evolution from Seq2Seq to Attention to Transformers represents a progression toward more effective handling of context, longer sequences, and parallel computation -- each building on the insights of its predecessor.